# Tutorial 2b &mdash; Cell&ndash;cell interaction: what are they saying to each other?

**ASI-FIMSA Workshop 2026 &mdash; spatial omics, hands-on**

Tutorial 2 established *who stands next to whom*. A T cell pressed against a carcinoma cell is
in a different world from one sitting in collagen 300 µm away. But adjacency is not
conversation. Two cells can touch and ignore each other, and the interesting question for an
immunologist is the next one: **is there evidence of signalling across that contact, and
between which populations?**

The standard answer is a **ligand&ndash;receptor** analysis. Take a curated list of known
ligand&ndash;receptor pairs, and look for places in the tissue where the ligand is expressed
in one cell and its receptor in the cells around it, more often than you would expect by
chance. We use [**stLearn**](https://stlearn.readthedocs.io/), whose ligand&ndash;receptor
test is *spatially constrained* &mdash; a pair only scores where ligand and receptor are
neighbours, not merely present in the same slide.

We work on the same 2,000 µm Crop as Tutorial 2, the same 16,006 Atera cells &mdash; but with
a different set of genes, and the first section is about why that swap was necessary at all.

| Section | The question | What you get back |
| --- | --- | --- |
| **1** | Why can't we do this on the Tutorial 2 file? | the cost of a panel choice, in pairs |
| **3** | How do you run a Visium-era tool on single cells? | a neighbourhood in micrometres |
| **5** | Which ligand&ndash;receptor pairs are spatially significant? | a ranked, permutation-tested list |
| **7** | Between which **cell types** does each pair act? | a sender &rarr; receiver matrix |

**What you will get out of it**

1. A concrete demonstration that **your gene panel decides which questions you can ask** &mdash;
   the same cells support 4 testable ligand&ndash;receptor pairs on one panel and 2,168 on
   another.
2. A permutation-tested list of spatially co-expressed ligand&ndash;receptor pairs, and the
   habit of reading the **top of that list sceptically**, because abundance drives it.
3. The immunology: **CD47&ndash;SIRPA**, **CSF1&ndash;CSF1R**, **CXCL12&ndash;CXCR4** and
   **ICAM1&ndash;ITGAL** localised to the cell-type pairs that carry them, on real tissue.

Everything runs on a free Colab CPU session. The two analysis cells in Sections 5 and 7 take
two to four minutes between them &mdash; the longest wait in the Workshop after Tutorial 3's
training loop. There is no GPU anywhere in this Tutorial.

> stLearn comes from the Genomics and Machine Learning lab at the University of Queensland.
> Its cell&ndash;cell interaction method is published as
> [Pham *et al.*, *Nature Communications* 2023](https://www.nature.com/articles/s41467-023-43120-6).
> This Tutorial is written against stlearn 1.4, whose API differs from the tutorials on the
> stLearn website in a few places; where it does, the cells below say so.

## 0. Setup

### 0.1 Install

Colab already ships **numpy**, **pandas**, **matplotlib**, **scikit-learn**, **scipy**,
**seaborn**, **torch** and **torchvision**. We add **stlearn** itself, **scanpy** (the AnnData
ecosystem underneath it), and two packages stlearn imports but does not install for us:
**bokeh** and **leidenalg**.

This takes about two minutes. Run it, then read Section 0.2 while it works.

> **Why stlearn is installed with `--no-deps`, and why that is a real decision.** stlearn 1.4.1
> declares that it needs `numpy>=2.4`. Colab ships **numpy 2.0.2** and imports it before your
> first cell runs, so letting pip satisfy that requirement would swap numpy underneath a
> running kernel and force a session restart &mdash; for a whole room, mid-Tutorial. Every
> other package stlearn imports is already installed by the line above, so we install stlearn
> alone, with its dependency resolution switched off, and hold numpy where Colab put it. That
> is an override of a declared incompatibility, and it is only known to be safe for the code
> this Tutorial actually runs. The Workshop verifies exactly these cells end to end before the
> day; if you take stlearn somewhere else, resolve its dependencies properly.

In [ ]:
# ============================================================================
# Install. You do not need to read or understand this cell -- just run it.
# ============================================================================
import subprocess
import sys
import urllib.request

IN_COLAB = "google.colab" in sys.modules

REPO_RAW = "https://raw.githubusercontent.com/xiao233333/ASI-FIMSA-workshop-2026/main"
PACKAGES = ["scanpy", "seaborn", "bokeh", "leidenalg"]

if IN_COLAB:
    args = list(PACKAGES)
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/constraints-colab.txt",
                                   "constraints-colab.txt")
        args = ["-c", "constraints-colab.txt"] + args
        print("using the Workshop pin set (constraints-colab.txt)")
    except Exception as exc:                       # noqa: BLE001
        print(f"could not fetch the pin set ({type(exc).__name__}); "
              "installing unpinned, which is usually fine")
    print("installing:", " ".join(PACKAGES), "... this takes about two minutes")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

    # stlearn imports torchvision at module level (for an image-feature extractor
    # this Tutorial never calls). Colab preinstalls it; install it only if it is
    # genuinely absent, so pip is never given a reason to re-resolve torch.
    try:
        import torchvision                          # noqa: F401
    except ImportError:
        print("torchvision is missing -- installing it")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchvision"],
                       check=False)

    # --no-deps: see the note above. numpy stays where Colab put it.
    print("installing: stlearn (--no-deps)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "stlearn"],
                   check=False)
    print("done")
else:
    print("Not running in Google Colab -- assuming stlearn, scanpy, seaborn, bokeh")
    print("and leidenalg are already installed in this environment.")

### 0.2 Imports

Two lines below are not ordinary imports, and both are worth a moment.

The first is a **thread guard**. Inside its analysis, stlearn calls
`numba.set_num_threads(os.cpu_count())`. On Colab those two numbers agree and nothing happens.
On any machine where a process is allowed fewer cores than the machine advertises &mdash; an
HPC job, a container with a CPU limit, most shared servers &mdash; `os.cpu_count()` overshoots
what numba will accept and the analysis dies with `ValueError: The number of threads must be
between 1 and N`. We reconcile the two numbers before stlearn looks at them.

The second is `import stlearn as st`, which is slow (five to fifteen seconds) because it pulls
in scanpy, spatialdata, bokeh and torch on the way. That is normal.

In [ ]:
import os
import time
import zipfile
from pathlib import Path

# ---- thread guard: run this BEFORE importing stlearn ------------------------
# stlearn's cci.run calls numba.set_num_threads(os.cpu_count()). Where the
# process is capped below the machine's core count, that raises. No-op on Colab.
import numba

N_THREADS = min(os.cpu_count() or 1, numba.config.NUMBA_NUM_THREADS)
os.cpu_count = lambda: N_THREADS          # noqa: E731  -- deliberate, see above
numba.set_num_threads(N_THREADS)

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import stlearn as st
from scipy.spatial import cKDTree

plt.rcParams["figure.dpi"] = 100

from importlib.metadata import version

for pkg in ("stlearn", "scanpy", "anndata", "numpy", "pandas", "numba"):
    print(f"{pkg:12s}: {version(pkg)}")
print(f"{'numba threads':12s}: {N_THREADS}")

try:
    import spatialdata as sd
    HAVE_SPATIALDATA = True
    print(f"{'spatialdata':12s}: {version('spatialdata')}")
except Exception as exc:                           # noqa: BLE001
    HAVE_SPATIALDATA = False
    print(f"spatialdata : not available ({type(exc).__name__}) -- "
          "the H&E backdrop in Section 6 will be skipped")

### 0.3 Download the data

Two prepared files, both derived from the same Atera slide as Tutorial 2:

* **`atera_crop_lr.h5ad`** (5 MB) &mdash; the analysis file. The same 16,006 Crop cells, with
  the same cell types and the same coordinates, but carrying **1,673 genes**: every
  connectomeDB2020 ligand or receptor that the Atera run measures and that is detected in at
  least ten of those cells. Counts are raw and stored sparse.
* **`atera_crop.zarr.zip`** (18 MB) &mdash; the Tutorial 2 Crop, used here only for its H&E
  image, so the interaction maps in Section 6 sit on tissue rather than on white space. If it
  fails to download, every figure still works and simply loses its backdrop.

The cell tries Hugging Face first, then Google Drive, and honours a `WORKSHOP_DATA_DIR`
environment variable if you already have the files locally.

In [ ]:
# ============================================================================
# Get atera_crop_lr.h5ad (required) and atera_crop.zarr.zip (optional backdrop)
# ============================================================================
HF_REPO = "xiao233333/asi-fimsa-workshop-2026"
LR_NAME = "atera_crop_lr.h5ad"
CROP_ZIP_NAME = "atera_crop.zarr.zip"
CROP_DIR = Path("atera_crop.zarr")
DATA_DIR = Path(os.environ.get("WORKSHOP_DATA_DIR", "."))
# Google Drive mirrors; the presenter sets these if Hugging Face is unreachable.
DRIVE_IDS = {
    LR_NAME: os.environ.get("WORKSHOP_LR_DRIVE_ID", ""),
    CROP_ZIP_NAME: os.environ.get("WORKSHOP_CROP_DRIVE_ID", ""),
}


def fetch(filename):
    """Local copy -> Hugging Face -> Google Drive. Returns a Path, or None."""
    for candidate in (DATA_DIR / filename, Path(filename)):
        if candidate.exists():
            print(f"{filename}: using local copy ({candidate})")
            return candidate
    try:
        from huggingface_hub import hf_hub_download
        p = Path(hf_hub_download(repo_id=HF_REPO, filename=filename,
                                 repo_type="dataset"))
        print(f"{filename}: downloaded from Hugging Face")
        return p
    except Exception as exc:                       # noqa: BLE001
        # Deliberately no traceback -- a wall of red text reads like something
        # you did wrong, and the Drive mirror below usually works.
        print(f"{filename}: Hugging Face not reachable ({type(exc).__name__})")
    if DRIVE_IDS.get(filename):
        try:
            import gdown
            gdown.download(id=DRIVE_IDS[filename], output=filename, quiet=True)
            if Path(filename).exists():
                print(f"{filename}: downloaded from the Google Drive mirror")
                return Path(filename)
        except Exception as exc:                   # noqa: BLE001
            print(f"{filename}: Drive mirror not reachable ({type(exc).__name__})")
    return None


lr_path = fetch(LR_NAME)
if lr_path is None:
    print()
    print("Could not fetch the analysis file. Please tell the presenter -- this is")
    print("our problem, not yours. If you have it already, put it next to this")
    print("notebook, or set WORKSHOP_DATA_DIR to the folder holding it, and re-run.")
else:
    print(f"  {lr_path.name}: {lr_path.stat().st_size / 1e6:.1f} MB")

crop_zip = fetch(CROP_ZIP_NAME)
if crop_zip is not None and not CROP_DIR.exists():
    # A .zarr.zip has to be unzipped before spatialdata.read_zarr() will open it.
    t0 = time.time()
    with zipfile.ZipFile(crop_zip) as zf:
        zf.extractall(CROP_DIR)
    print(f"  unzipped to {CROP_DIR}/ in {time.time() - t0:.1f} s")
HAVE_HE = CROP_DIR.exists() and HAVE_SPATIALDATA

---

## 1. The same cells, a different panel

This is the same Crop you worked on in Tutorial 2. Same tissue, same 16,006 cells, same cell
type per cell, same coordinates in micrometres. The only thing that changed is which genes came
along.

In [ ]:
adata = ad.read_h5ad(lr_path)
print(adata)
print()
print("coordinates (obsm['spatial']), in micrometres:")
print("  x:", adata.obsm["spatial"][:, 0].min().round(0), "to",
      adata.obsm["spatial"][:, 0].max().round(0))
print("  y:", adata.obsm["spatial"][:, 1].min().round(0), "to",
      adata.obsm["spatial"][:, 1].max().round(0))
print()
print("cell types:")
print(adata.obs["cell_type"].value_counts().to_string())

### 1.1 Why Tutorial 2's file could not be used

Tutorial 2's Crop carries **69 genes**, chosen to name cell types: `EPCAM`, `KRT8`, `CD3D`,
`CD68`, `COL1A1` and so on. That panel is excellent at its job and useless at this one.

A ligand&ndash;receptor test needs **both halves of a pair** to be measured. Intersect
connectomeDB2020's 2,293 literature-supported pairs with those 69 genes and you are left with
four, two of which are the same pair read in both directions. There is nothing to analyse.

The genes were never missing from the experiment. Atera is whole-transcriptome &mdash; 18,028
targets &mdash; so 1,675 of connectomeDB's ligands and receptors were measured on this slide
all along. They simply were not carried into the file Tutorial 2 needed. The file below carries
them instead.

In [ ]:
cov = adata.uns["atera"]["lr_source"]["coverage"]
rows = []
for db, c in cov.items():
    rows.append({
        "database": db.replace("connectomeDB2020_", ""),
        "pairs in database": c["pairs_in_database"],
        "measurable by Atera": c["pairs_in_atera"],
        "in this 1,673-gene panel": c["pairs_in_this_panel"],
        "in Tutorial 2's 69 genes": c["pairs_in_teaching_panel"],
    })
coverage = pd.DataFrame(rows).set_index("database")
print(coverage.to_string())

fig, ax = plt.subplots(figsize=(7.6, 3.4))
y = np.arange(len(coverage))
ax.barh(y - 0.2, coverage["in Tutorial 2's 69 genes"], height=0.38,
        color="#bdbdbd", label="Tutorial 2 panel (69 genes)")
ax.barh(y + 0.2, coverage["in this 1,673-gene panel"], height=0.38,
        color="#1f6fb4", label="this panel (1,673 genes)")
for yi, v in zip(y - 0.2, coverage["in Tutorial 2's 69 genes"]):
    ax.text(v + 60, yi, str(v), va="center", fontsize=9)
for yi, v in zip(y + 0.2, coverage["in this 1,673-gene panel"]):
    ax.text(v + 60, yi, f"{v:,}", va="center", fontsize=9)
ax.set_yticks(y, coverage.index)
ax.set_xlabel("complete ligand-receptor pairs (both genes measured)")
ax.set_title("The same 16,006 cells. What each panel can even ask.")
ax.legend(fontsize=8, loc="lower right")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

Four against 2,168. That is the entire content of a panel decision, made months earlier by
someone choosing which probes to order, arriving as a hard limit on what you are allowed to
conclude.

> **The general lesson.** Targeted spatial platforms &mdash; Xenium, CosMx, MERFISH &mdash;
> measure the genes on their panel and nothing else. A 300-gene immuno-oncology panel will
> support cell typing beautifully and ligand&ndash;receptor analysis barely, because curated
> interaction databases are built from the whole transcriptome and your panel is not.
> **Before you buy a panel, take the analysis you intend to run and intersect it with the probe
> list.** It takes ten minutes and it is the cheapest experiment you will ever do. The Atera
> chemistry used here exists precisely to remove this constraint: whole-transcriptome, at
> single-cell resolution, so the panel stops being the thing that decides.

### 1.2 What is in the file

`var` records what each gene is for. A gene can be a ligand in one pair and a receptor in
another &mdash; `PTPRC` and `CD47` both are &mdash; so the roles are not exclusive.

In [ ]:
print(adata.var["role"].value_counts().to_string())
print()
print(adata.var.sort_values("n_cells_detected", ascending=False).head(10).to_string())
print()
print("read_instructions:")
print(" ", adata.uns["atera"]["read_instructions"])

---

## 2. What a ligand&ndash;receptor test actually asks

The pair list is not data. It is a **curated hypothesis set** &mdash; somebody read the
literature and decided that these molecules bind. stlearn ships two versions of
connectomeDB2020, and the difference between them is the difference between two standards of
evidence:

| Database | Pairs | What it means |
| --- | --- | --- |
| `connectomeDB2020_lit` | 2,293 | each pair has literature support for the interaction |
| `connectomeDB2020_put` | 4,071 | the above, plus *putative* pairs inferred from protein-domain evidence |

We use the literature set. The putative set finds more, and some of what it finds is real, but
a hit you cannot trace to a paper is a hypothesis rather than a result &mdash; and with 4,071
pairs the multiple-testing burden nearly doubles.

`load_lrs` reads these from inside the stlearn package, so no download is involved.

In [ ]:
lrs_lit = st.tl.cci.load_lrs(["connectomeDB2020_lit"], species="human")
lrs_put = st.tl.cci.load_lrs(["connectomeDB2020_put"], species="human")
print(f"connectomeDB2020_lit : {len(lrs_lit):,} pairs")
print(f"connectomeDB2020_put : {len(lrs_put):,} pairs")
print()
print("a pair is a single string, 'LIGAND_RECEPTOR':")
print(" ", ", ".join(lrs_lit[:6]))

# Keep only the pairs whose ligand AND receptor are both in our panel.
measured = set(adata.var_names)
candidates = np.array([p for p in lrs_lit if all(g in measured for g in p.split("_"))])
print()
print(f"{len(candidates):,} of the {len(lrs_lit):,} literature pairs are fully measured here")

immune = ["CD47_SIRPA", "CSF1_CSF1R", "CXCL12_CXCR4", "ICAM1_ITGAL", "TGFB1_TGFBR2",
          "B2M_HLA-F", "SPP1_CD44", "VCAM1_ITGA4", "IL16_CD4", "LGALS1_PTPRC"]
print("some pairs an immunologist will recognise, and whether we can test them:")
for p in immune:
    print(f"  {p:16s} {'yes' if p in set(candidates) else 'NO'}")

`CD47&ndash;SIRPA` is the "don't eat me" axis that anti-CD47 antibodies target.
`CSF1&ndash;CSF1R` recruits and sustains tumour-associated macrophages and is itself a drug
target. `CXCL12&ndash;CXCR4` structures lymphocyte positioning throughout the tumour
microenvironment. All three are testable here and none of them were testable in Tutorial 2.

---

## 3. Teaching a Visium tool to read single cells

stlearn was written in the Visium era, when a "spot" was a 55 µm circle containing several
cells arranged on a fixed hexagonal grid. Our data is single cells at arbitrary positions. The
method transfers &mdash; a neighbourhood is a neighbourhood &mdash; but two of its conventions
have to be met explicitly, and both are the kind of thing that costs an afternoon if nobody
tells you.

**First, stlearn reads positions from `obs["imagerow"]` and `obs["imagecol"]`,** not from
`obsm["spatial"]` where every other tool in the scverse ecosystem looks. Nothing warns you: the
`KeyError` arrives several function calls deep. The staged file deliberately does *not* ship
these columns, so you have to write the line yourself and see what it is.

**Second, `distance` must be given explicitly.** Left to itself, stlearn computes the
neighbourhood radius from `uns["spatial"][...]["scalefactors"]["spot_diameter_fullres"]`
&mdash; a Visium field our data does not have and should not have. Because our coordinates are
in micrometres, we can pass something biologically meaningful instead.

In [ ]:
# stlearn looks here, not in obsm["spatial"]. col = x, row = y.
adata.obs["imagecol"] = adata.obsm["spatial"][:, 0].astype(float)
adata.obs["imagerow"] = adata.obsm["spatial"][:, 1].astype(float)

# How many neighbours does each radius actually buy? Coordinates are micrometres,
# and these cells are ~10-15 um across, so the answer is interpretable.
tree = cKDTree(adata.obsm["spatial"])
rows = []
for d in (10, 15, 20, 30, 40, 60):
    k = np.array([len(n) - 1 for n in tree.query_ball_point(adata.obsm["spatial"], d)])
    rows.append({"radius (um)": d, "median neighbours": int(np.median(k)),
                 "mean": round(k.mean(), 1),
                 "% with none": round(100 * (k == 0).mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))

Below 15 µm a large fraction of cells have no neighbour at all and simply cannot score. Above
40 µm the "neighbourhood" spans several cell diameters and the spatial constraint starts to
dissolve &mdash; at which point you are asking whether two genes are expressed in the same
*region*, which is a weaker claim than whether they are expressed in touching cells.

**We use 30 µm**: a median of 14 neighbours, and 0.13% of cells isolated. Look at what that
radius means on the tissue before trusting it.

In [ ]:
NEIGHBOUR_UM = 30.0

# Draw the neighbourhood graph on a small window, exactly as Tutorial 2 does for
# its Delaunay graph -- a summary statistic is not a substitute for looking.
from matplotlib.collections import LineCollection

xy = adata.obsm["spatial"]
# The Crop knows where its most cell-type-diverse 300 um square is -- it was
# picked at build time, and it is the window Tutorial 2 zoomed into as well.
zoom = adata.uns["atera"]["crop_window"]["he_zoom"]
w0, h0, size = zoom["x0_um"], zoom["y0_um"], zoom["size_um"]
sel = np.flatnonzero((xy[:, 0] >= w0) & (xy[:, 0] < w0 + size)
                     & (xy[:, 1] >= h0) & (xy[:, 1] < h0 + size))
sub_xy = xy[sel]
pairs = cKDTree(sub_xy).query_pairs(NEIGHBOUR_UM, output_type="ndarray")
segs = [(sub_xy[i], sub_xy[j]) for i, j in pairs]

types = list(adata.obs["cell_type"].cat.categories)
PALETTE = dict(zip(types, adata.uns["cell_type_colors"]))
colours = [PALETTE[t] for t in adata.obs["cell_type"].to_numpy()[sel]]

fig, ax = plt.subplots(figsize=(8.4, 7.2))
ax.add_collection(LineCollection(segs, colors="0.78", linewidths=0.4, zorder=1))
ax.scatter(sub_xy[:, 0], sub_xy[:, 1], s=30, c=colours, lw=0.3,
           edgecolor="white", zorder=2)
sub_types = adata.obs["cell_type"].to_numpy()[sel]
for t in [t for t in types if (sub_types == t).any()]:
    ax.scatter([], [], s=40, c=PALETTE[t], label=f"{t} ({int((sub_types == t).sum())})")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")
ax.set_title(f"the {NEIGHBOUR_UM:.0f} µm neighbourhood — {len(sel):,} cells and "
             f"{len(segs):,} edges in a {size:.0f} µm square")
plt.tight_layout()
plt.show()

That is the object every score in this Tutorial is computed over. Cells in a dense tumour nest
have many neighbours; a fibroblast in open stroma has few, and will find it correspondingly
harder to reach significance. That is not a bug &mdash; it is the honest consequence of asking
a question about contact &mdash; but it does mean **the method has more power in dense tissue**,
and you should keep that in mind when a sparse compartment comes back quiet.

### 3.1 Normalise, then densify

Two housekeeping steps. The counts arrive raw, so we normalise per cell and `log1p` as usual.
Then we make the matrix dense, because stlearn's internals call `adata.to_df()`, which cannot
work from a sparse matrix without materialising it anyway.

In [ ]:
n_zero = int((np.asarray(adata.X.sum(axis=1)).ravel() == 0).sum())
print(f"{n_zero} cells have zero counts across the whole LR panel "
      f"({100 * n_zero / adata.n_obs:.2f}%) -- they can never score")

sc.pp.normalize_total(adata, target_sum=1e2)
sc.pp.log1p(adata)

# stlearn's get_spot_lrs() calls adata.to_df(); sparse buys nothing downstream.
adata.X = np.asarray(adata.X.todense(), dtype=np.float32)
print(f"X is now {adata.X.shape} dense float32 "
      f"({adata.X.nbytes / 1e6:.0f} MB in memory)")

---

## 4. Choosing which pairs to test

We could hand all 2,168 pairs to the test. We do not, for two reasons.

The first is time: the permutation test builds a fresh null distribution for *every* pair, so
cost is linear in the number of pairs, and 2,168 of them would run for a quarter of an hour on
a Colab CPU.

The second matters more. A pair whose ligand is detected in forty cells out of sixteen thousand
cannot reach significance no matter how real the biology is; it only adds to the multiple-
testing burden and pushes down everything else. So we keep pairs where **both genes are
detected in at least 5% of cells** &mdash; a rule about statistical opportunity, applied before
the expensive step rather than after it.

> **The genes you leave out are not wasted.** stlearn builds its null by drawing random gene
> pairs with *similar expression* to the pair under test, from every gene in `var` that is not
> in the tested set. Trimming to 402 pairs leaves 1,368 genes in that background pool, which is
> a healthier null than testing everything and having almost nothing left to draw from.

In [ ]:
MIN_DETECTED_FRAC = 0.05
min_cells = int(round(MIN_DETECTED_FRAC * adata.n_obs))

detected = adata.var["n_cells_detected"]
opportunity = np.array([min(detected[p.split("_")[0]], detected[p.split("_")[1]])
                        for p in candidates])
lrs = candidates[opportunity >= min_cells]

tested_genes = {g for p in lrs for g in p.split("_")}
print(f"threshold           : both genes in >= {min_cells:,} cells "
      f"({MIN_DETECTED_FRAC:.0%} of {adata.n_obs:,})")
print(f"pairs to test       : {len(lrs):,} of {len(candidates):,}")
print(f"genes they use      : {len(tested_genes):,}")
print(f"background pool     : {adata.n_vars - len(tested_genes):,} genes")
print()
print("immunology that survives the cut:")
print(" ", ", ".join(p for p in immune if p in set(lrs)))

Every pair we flagged in Section 2 clears the threshold. Note what the rule is *not* doing: it
does not know which pairs are interesting, only which are testable. Section 5.1 is about what
that costs.

---

## 5. Running the test

For each ligand&ndash;receptor pair and each cell, stlearn computes a score that is high when
the ligand is expressed in that cell and the receptor in its neighbours (and the reverse), and
zero when either half is missing. It then asks whether that score is larger than the scores of
random gene pairs matched on expression level &mdash; a per-cell, per-pair permutation test,
FDR-corrected across cells.

Three numbers control it:

| Argument | Here | What it does |
| --- | --- | --- |
| `distance` | `30.0` | neighbourhood radius, in the units of `imagerow`/`imagecol` &mdash; micrometres |
| `min_spots` | `20` | a pair scoring in fewer than 20 cells is dropped before permutation |
| `n_pairs` | `200` | how many random pairs to build each null from |

> **`n_pairs=200` is a compromise with the clock, and you should know it.** stlearn's own
> documentation recommends 10,000 for a real analysis. At 200 the null is coarse, so the
> smallest p-value the test can resolve is bounded and the ranking near the bottom of the list
> is noisy. It is enough to see the structure, and it is not enough for a figure in a paper.
> When you run this on your own data, set it to 10,000, start it, and go to lunch.

This cell takes one to two minutes. The progress bar is stlearn's.

In [ ]:
MIN_SPOTS = 20
N_BACKGROUND_PAIRS = 200

t0 = time.time()
st.tl.cci.run(
    adata,
    lrs,
    min_spots=MIN_SPOTS,
    distance=NEIGHBOUR_UM,
    n_pairs=N_BACKGROUND_PAIRS,
    verbose=True,
)
print(f"\ncci.run finished in {time.time() - t0:.1f} s")

lr_summary = adata.uns["lr_summary"]
print(f"{lr_summary.shape[0]:,} pairs survived min_spots={MIN_SPOTS}")
print(lr_summary.head(15).to_string())

`n_spots` is how many cells the pair scored in at all; `n_spots_sig` is how many of those
survived FDR correction. The per-cell numbers live in `adata.obsm` &mdash; `lr_scores`,
`p_adjs`, `lr_sig_scores` &mdash; with columns in the same order as the rows of `lr_summary`.

### 5.1 Read the top of the list sceptically

Before interpreting anything, look at what the ranking is made of.

In [ ]:
# stlearn's own ranking plot. It manages its own figure, so it gets its own cell
# block -- mixing it into a plt.subplots grid leaves the other panel empty.
st.pl.lr_summary(adata, n_top=25, figsize=(9, 4.2))
plt.show()

In [ ]:
# The same ranking, coloured by what kind of biology each pair is.
MATRIX_LIGANDS = ("COL", "LAM", "FN1", "THBS", "HSPG", "FBN", "CCN", "POSTN",
                  "TNC", "VIM", "SPP1", "NID", "VTN", "AGRN", "ELN", "LTBP")
MATRIX_RECEPTORS = {"CD44", "LRP1", "SDC1", "SDC2", "SDC4", "PTK7"}


def is_matrix_adhesion(pair):
    """Matrix protein talking to an integrin or a matrix receptor."""
    ligand, receptor = pair.split("_")
    return (ligand.startswith(MATRIX_LIGANDS)
            or receptor.startswith("ITG")
            or receptor in MATRIX_RECEPTORS)


top25 = lr_summary.head(25).index
flag = [is_matrix_adhesion(p) for p in top25]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(top25)), lr_summary.loc[top25, "n_spots_sig"],
        color=["#8c6d3f" if f else "#1f6fb4" for f in flag])
ax.set_yticks(range(len(top25)), top25, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("cells with a significant score")
ax.set_title(f"brown = matrix / adhesion ({sum(flag)} of {len(top25)}), "
             "blue = everything else")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

Twenty-one of the top twenty-five are **extracellular matrix meeting an integrin**:
`COL4A1_ITGAV`, `LAMB1_ITGB1`, `THBS1_ITGB1`, `FN1_ITGB1`, `HSPG2_ITGB1`, `FBN1_ITGAV`. Collagen,
laminin, fibronectin, thrombospondin, perlecan, fibrillin &mdash; the structural fabric of the
tumour, binding the receptors that hold cells to it.

This is a true result and a boring one, and it matters that you understand why it happens.
Those genes are among the most abundantly expressed transcripts in any solid tumour, so they
are detected in the most cells, so they have the most opportunity to score. The ranking is
reporting **statistical power**, which tracks abundance, at least as much as it is reporting
biology. The pair at number one, `B2M_HLA-F`, is not matrix but wins for exactly the same
reason: `B2M` is detected in 13,390 of the 16,006 cells &mdash; more than any other gene in the
panel.

> **This is the single most common way to misread a ligand&ndash;receptor analysis.** The top
> of the list is not "the most important signalling in this tumour". It is "the pairs with
> enough expression to clear a significance threshold". If your question is about a specific
> axis &mdash; and an immunologist's usually is &mdash; go and look at that axis directly
> rather than waiting for it to appear at the top of a ranked list. Section 7.2 does exactly
> that.

### 5.2 Diagnostics

`lr_diagnostics` plots the two things that determine whether the null was fair: the median
non-zero expression of each pair, and the proportion of cells where it scores zero, both
against the pair's rank.

In [ ]:
# Both of these build and show their own figure -- calling plt.tight_layout()
# afterwards would open a second, empty one.
st.pl.lr_diagnostics(adata, figsize=(11, 2.8))
plt.show()

st.pl.lr_n_spots(adata, n_top=25, figsize=(9, 5.5))
plt.show()

The left panel falls and the right panel rises with rank, which is exactly the expected shape:
highly-ranked pairs are more expressed and zero in fewer cells. What you are checking for is a
*discontinuity* &mdash; a cluster of top hits that sit in a different expression regime from
everything else would mean the background matching failed for them.

The second figure splits each bar into cells that reached significance and cells that did not.
A pair scoring in thousands of cells but significant in few is spatially diffuse: co-expressed
everywhere, structured nowhere.

---

## 6. Where in the tissue?

A ranked table is not a spatial result. The whole point of doing this on tissue is that a pair
can be significant in one region and silent in another, so put the per-cell scores back on the
slide.

stlearn's own spatial plotting functions (`st.pl.lr_plot`, `st.pl.lr_result_plot`) expect
`adata.uns["spatial"]` in Visium's format &mdash; a library id, a hires image, and scalefactors
&mdash; and raise if it is missing. Rather than fabricate that structure, we draw these
ourselves with matplotlib, exactly as Tutorial 2 does, over the Crop's own H&E.

In [ ]:
# The same helper as Tutorial 2: an RGB array plus an extent in micrometres,
# taken from the element's own transformation, so nobody hard-codes a pixel size.
HE, HE_EXTENT = None, None
if HAVE_HE:
    from spatialdata.transformations import get_transformation

    sdata = sd.read_zarr(CROP_DIR)
    el = sdata["he"]
    HE = np.asarray(el.transpose("y", "x", "c").data)
    M = get_transformation(el, "global").to_affine_matrix(
        input_axes=("x", "y"), output_axes=("x", "y"))
    h, w = HE.shape[:2]
    (x0, x1), (y0, y1) = (M @ np.array([[0, 0, 1], [w, h, 1]]).T)[:2]
    HE_EXTENT = (x0, x1, y1, y0)          # y flipped: images draw top-down
    print(f"H&E backdrop {HE.shape} covering {HE_EXTENT}")
else:
    print("no H&E available -- the maps below will be drawn on white")


def score_map(lr, ax, sig_only=True, cmap="viridis", title=None):
    """Draw one LR pair's per-cell score over a desaturated H&E.

    The H&E goes to greyscale on purpose. Score colour against pink-and-purple
    tissue is unreadable, and the backdrop is here for anatomical context, not
    for its own sake.
    """
    key = "lr_sig_scores" if sig_only else "lr_scores"
    j = list(lr_summary.index).index(lr)
    v = np.asarray(adata.obsm[key])[:, j]
    if HE is not None:
        grey = HE.mean(axis=2)
        # Stretch the contrast: H&E converted to grey sits in a narrow bright
        # band, and vmin=0/vmax=255 renders the tissue as near-white paper.
        ax.imshow(grey, extent=HE_EXTENT, cmap="gray", alpha=0.9,
                  vmin=np.percentile(grey, 2), vmax=np.percentile(grey, 98))
    hit = v > 0
    ax.scatter(xy[~hit, 0], xy[~hit, 1], s=0.7, c="0.8", lw=0, alpha=0.25)
    pts = ax.scatter(xy[hit, 0], xy[hit, 1], s=9, c=v[hit], cmap=cmap, lw=0,
                     vmin=0, vmax=np.percentile(v[hit], 98) if hit.any() else 1)
    ax.set_aspect("equal")
    if HE_EXTENT is not None:
        ax.set_xlim(HE_EXTENT[0], HE_EXTENT[1])
        ax.set_ylim(HE_EXTENT[2], HE_EXTENT[3])
    else:
        ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{title or lr}\n{int(hit.sum()):,} significant cells", fontsize=10)
    return pts


top_lr = lr_summary.index[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (lr, label) in zip(axes, [(top_lr, f"{top_lr} — the top-ranked pair"),
                                  ("CD47_SIRPA", "CD47_SIRPA — the 'don\'t eat me' axis")]):
    pts = score_map(lr, ax, title=label)
    fig.colorbar(pts, ax=ax, shrink=0.62, label="ligand-receptor score")
plt.tight_layout()
plt.show()

**The obvious difference is density, not location.** `B2M_HLA-F` reaches significance in 1,539
cells and `CD47_SIRPA` in 430 &mdash; a factor of 3.6, and precisely the abundance effect
Section 5.1 warned about, now visible as dots on tissue.

**The less obvious thing is that both maps avoid the same places.** The dark, densely nucleated
masses are carcinoma nests, and their interiors are sparse in both panels; the signal sits in
the paler interstitial stroma between and around them. Rather than take that from a figure, ask
the labels.

In [ ]:
# Which cell types actually carry the significant scores, against their share of
# the whole Crop? A ratio of 1.0 means "exactly as often as chance".
def enrichment(lr):
    j = list(lr_summary.index).index(lr)
    hit = np.asarray(adata.obsm["lr_sig_scores"])[:, j] > 0
    return adata.obs["cell_type"][hit].value_counts(normalize=True) * 100


overall = adata.obs["cell_type"].value_counts(normalize=True) * 100
enr = pd.DataFrame({"% of Crop": overall,
                    f"% of {top_lr} cells": enrichment(top_lr),
                    "% of CD47_SIRPA cells": enrichment("CD47_SIRPA")}).fillna(0)
enr["fold (top pair)"] = enr.iloc[:, 1] / enr["% of Crop"]
enr["fold (CD47_SIRPA)"] = enr.iloc[:, 2] / enr["% of Crop"]
print(enr.round(2).sort_values("fold (CD47_SIRPA)", ascending=False).to_string())

The table says it far more sharply than the figure does.

**Tumour epithelial cells are 42.7% of the Crop and 3.5% of the cells carrying a significant
`B2M_HLA-F` score** &mdash; a twelve-fold depletion. For `CD47_SIRPA` it is six-fold. Every
immune population runs the other way: T cells are 3.7&times; enriched for `B2M_HLA-F`, and
macrophages are **3.9&times; enriched for `CD47_SIRPA`**, which is the cleanest possible
sanity check, because SIRPα is a myeloid receptor and that is exactly where CD47 signalling
should be read out.

Two cautions before you enjoy that too much. Mast cells score 4.5&times; on 33 cells, which is
noise wearing a large number. And a ligand&ndash;receptor score needs *two different things* in
one neighbourhood, so a solid sheet of one cell type is structurally disadvantaged &mdash;
some of the tumour depletion is the method, not the biology.

Even allowing for that, compare those interstitial bands with Tutorial 2's immune-infiltration
niche. They are the same regions, reached from completely different directions: Tutorial 2 from
*which cell types are near each other*, this Tutorial from *which transcripts are near each
other*. When two independent analyses of one tissue agree on where the action is, the structure
is probably real &mdash; and the quiet interior of a carcinoma nest is the same
immune-exclusion story, told twice.

---

## 7. From pairs to cell types

So far every result has been per *cell*. The question an immunologist actually asks is per
*population*: **which cell type is sending, and which is receiving?**

`run_cci` answers it. For every significant cell and every pair, it looks at the cell's own
type and the types of the neighbours carrying the other half of the pair, counts
sender&nbsp;&rarr;&nbsp;receiver events, and permutes the cell type labels to ask whether that
count is more than chance would give. `use_label="cell_type"` means our labels are used
directly &mdash; on Visium you would pass deconvolution proportions instead, because a spot is
a mixture.

This cell takes one to two minutes.

In [ ]:
N_PERMS = 100

t0 = time.time()
st.tl.cci.run_cci(
    adata,
    use_label="cell_type",
    min_spots=3,
    n_perms=N_PERMS,
    verbose=True,
)
print(f"\nrun_cci finished in {time.time() - t0:.1f} s")

cci = adata.uns["lr_cci_cell_type"]
print(f"\nsender -> receiver interaction counts, {cci.shape[0]} x {cci.shape[1]}")

In [ ]:
st.pl.cci_map(adata, "cell_type", figsize_or_none=(7.5, 6.5),
              title="all pairs: sender (row) -> receiver (column)")
plt.show()

hetero = cci.copy()
np.fill_diagonal(hetero.values, 0)
top = hetero.stack().sort_values(ascending=False).head(15)[::-1]
labels = [f"{sender} → {receiver}" for sender, receiver in top.index]
IMMUNE_TYPES = ("T cell", "Dendritic cell", "Macrophage", "Plasma cell", "Mast",
                "Mixed (plasma+mast)")
is_immune = [any(t in lab for t in IMMUNE_TYPES) for lab in labels]

fig, ax = plt.subplots(figsize=(8.5, 5.6))
ax.barh(range(len(top)), top.values,
        color=["#1f6fb4" if im else "#bdbdbd" for im in is_immune])
ax.set_yticks(range(len(top)), labels, fontsize=9)
ax.set_xlabel("interactions (significant sender-receiver events)")
ax.set_title("top 15 between different cell types — blue involves an immune cell")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

The diagonal dominates the heatmap, and it should: a tumour cell's nearest neighbour is
overwhelmingly another tumour cell, so tumour&nbsp;&rarr;&nbsp;tumour accumulates the most
events. Zeroing it out to look at cross-population signalling gives a much more informative
ranking.

What comes back is the tumour&ndash;stroma architecture:
**myoepithelial&nbsp;&harr;&nbsp;tumour epithelial** and
**fibroblast&nbsp;&harr;&nbsp;myoepithelial** at the top, then the vascular pairs
(endothelial&nbsp;&harr;&nbsp;perivascular). Myoepithelium is the layer that separates
epithelium from stroma in breast tissue, so it appearing on both sides of the busiest
interfaces is the analysis rediscovering the histology &mdash; a reassuring sign that the
method is picking up real tissue organisation rather than noise.

And then, high in the same list: **T&nbsp;cell&nbsp;&harr;&nbsp;fibroblast** and
**T&nbsp;cell&nbsp;&harr;&nbsp;dendritic&nbsp;cell**.

### 7.1 The network and the chord diagram

Two views of the same matrix. The network sizes each node by how much traffic it carries; the
chord diagram makes the balance of sending and receiving legible.

In [ ]:
st.pl.ccinet_plot(adata, "cell_type", min_counts=4000, figsize=(9, 9),
                  node_size_scaler=2)
plt.show()

st.pl.lr_chord_plot(adata, "cell_type")
plt.show()

> **`min_counts` is a readability knob, not a statistical one.** Without it the network is
> nearly complete &mdash; thirteen cell types, most pairs carrying some traffic &mdash; and
> a complete graph communicates nothing. Raising the threshold hides weak edges; it does not
> make the strong ones stronger. Say what you set it to whenever you show one of these.

### 7.2 Ask the immunology directly

Section 5.1 argued that the top of a ranked list reports abundance. So do not read down the
list &mdash; go straight to the pairs you care about and ask where each one acts. Every pair
below is one an immunologist would name unprompted, and `per_lr_cci_cell_type` holds a full
sender&nbsp;&rarr;&nbsp;receiver matrix for each.

In [ ]:
per_lr = adata.uns["per_lr_cci_cell_type"]
watch = [p for p in ["CD47_SIRPA", "CSF1_CSF1R", "CXCL12_CXCR4", "ICAM1_ITGAL",
                     "TGFB1_TGFBR2", "IL16_CD4", "VCAM1_ITGA4", "B2M_HLA-F"]
         if p in per_lr]

rows = []
for p in watch:
    m = per_lr[p].copy()
    np.fill_diagonal(m.values, 0)
    s = m.stack().sort_values(ascending=False)
    rows.append({
        "pair": p,
        "significant cells": int(lr_summary.loc[p, "n_spots_sig"]),
        "strongest sender -> receiver": f"{s.index[0][0]} → {s.index[0][1]}",
        "n": int(s.iloc[0]),
        "runner-up": f"{s.index[1][0]} → {s.index[1][1]}",
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14.5, 11.5))
for ax, pair in zip(axes.ravel(), ["CD47_SIRPA", "CSF1_CSF1R",
                                   "CXCL12_CXCR4", "ICAM1_ITGAL"]):
    m = per_lr[pair].copy()
    np.fill_diagonal(m.values, 0)          # same-type traffic swamps everything
    keep = m.index[(m.sum(axis=1) + m.to_numpy().sum(axis=0)) > 0]
    m = m.loc[keep, keep]
    # One or two cell-type pairs carry most of the counts, so cap the colour
    # scale at the 97th percentile of the non-zero entries; anything above it
    # saturates rather than flattening everything else to white.
    nz = m.to_numpy()[m.to_numpy() > 0]
    sns.heatmap(m, cmap="magma_r", ax=ax, square=True,
                vmin=0, vmax=np.percentile(nz, 97) if nz.size else 1,
                cbar_kws={"shrink": 0.62, "label": "interactions"},
                linewidths=0.3, linecolor="white")
    ax.set_title(f"{pair}   (diagonal removed)", fontsize=11)
    ax.set_xlabel("receiver")
    ax.set_ylabel("sender")
    ax.tick_params(labelsize=7)
plt.tight_layout()
plt.show()

Read the table and the four panels together.

**One axis dominates the immune half of this tumour: T cell &harr; dendritic cell.** It is the
top or the runner-up cross-population signal for six of the eight pairs above &mdash;
`ICAM1_ITGAL`, `TGFB1_TGFBR2`, `IL16_CD4`, `VCAM1_ITGA4`, `B2M_HLA-F`, and `CD47_SIRPA` in
second place. `ICAM1_ITGAL` is ICAM-1 binding LFA-1, the adhesion step of the immunological
synapse; `IL16_CD4` and `B2M_HLA-F` are antigen-presentation-adjacent. Finding all of them
concentrated on the same pair of populations is coherent: this is what T cells meeting
antigen-presenting cells looks like when you can only see transcripts.

**`CD47_SIRPA`** &mdash; the "don't eat me" signal, and the target of a whole class of
immuno-oncology drugs. Its strongest cross-population signal here is **myoepithelial &rarr;
tumour epithelial** (124 events), with T cell &harr; dendritic cell just behind (103 and 95).
So CD47 is doing two jobs in this tissue: shielding the epithelial compartment, and its
physiological role of restraining phagocytosis inside the immune compartment.

**`CSF1_CSF1R`** &mdash; how macrophages are recruited and kept alive. It is the weakest of the
four (241 significant cells) and its top sender is **macrophage &rarr; fibroblast**, not
tumour &rarr; macrophage. Read that cautiously rather than as a finding: `CSF1` sits close to
the detection floor, so this is exactly the regime where a sparse single-cell count matrix is
least trustworthy.

**`CXCL12_CXCR4`** &mdash; the chemokine gradient that positions lymphocytes, and the one pair
here with a clear stromal partner: **T cell &harr; fibroblast** (355 events, with 153 back the
other way). That is the T-cell-in-collagen picture from Tutorial 2, now with a candidate
molecule attached to it.

> **Before you believe the T cell &harr; dendritic cell story, apply Section 5.1's own
> argument to it.** T cells (1,590) and dendritic cells (928) are the two most numerous immune
> populations in this Crop; macrophages number 415 and plasma cells 91. More cells means more
> neighbour pairs means more opportunity to reach significance, so abundance is a live
> alternative explanation here too. What would distinguish them is a pair that is *specific*
> to a rarer population despite its size &mdash; and `CSF1_CSF1R` reaching significance at all
> with 415 macrophages is a small piece of evidence in that direction.

Note also what is **absent**. There is no B cell population in this Crop at all &mdash;
Tutorial 2 Section 1.4 went into why &mdash; so every B-cell axis is unanswerable here, no
matter how many genes we carried. **A missing cell type is a harder limit than a missing
gene**: Section 1 fixed the second problem and could do nothing about the first.

---

## 8. What this does not show

Every figure above is a statement about **transcript co-location**. It is worth being precise
about the distance between that and "these cells are signalling".

* **mRNA is not protein.** A cell transcribing `CSF1R` may not display the receptor; a
  displayed receptor may not be engaged. Nothing here measures binding.
* **Neighbourhood is not contact.** Two cells 25 µm apart are neighbours by our radius and are
  not touching. The 30 µm choice is defensible; it is still a choice, and every result in this
  Tutorial moves if you change it.
* **The pair list is somebody's literature review.** connectomeDB2020 knows what its curators
  read. A genuine interaction that is not in the list cannot be found, and a pair in the list
  that does not occur in breast tissue can still come back significant.
* **The ranking follows abundance.** Section 5.1 &mdash; the loudest pairs are the
  best-expressed ones. Immune signalling is transcriptionally quiet relative to matrix
  production, and will essentially never top this list.
* **`n_pairs=200` is a coarse null.** Set it to 10,000 before you believe a specific p-value.
* **Sparse counts, single cells.** A Xenium-class cell carries a few hundred transcripts across
  1,673 genes here. Most cell-by-gene entries are zero, and a pair only scores where both halves
  happen to be detected. Absence of signal is weak evidence of absence of signalling.
* **One Crop, one specimen.** 2,000 µm of one tumour, with no replicates. Everything here is a
  hypothesis generated on a single field of view.

None of this makes the analysis useless. It makes it a **hypothesis generator with
coordinates** &mdash; which is a great deal more than a bulk experiment can give you, and a
great deal less than proof.

---

## 9. Exercises

Each of these is a small edit to a cell above, followed by re-running from that cell down.

**1. Change the neighbourhood.** `NEIGHBOUR_UM = 30.0` in Section 3. Re-run Sections 5 and 7
with `15` and with `60`. How much of the top-20 list survives all three? A pair that only
appears at 60 µm is telling you about regional co-expression, not about contact &mdash; treat
the pairs stable across all three as your real findings.

**2. Ask an immune question instead of an abundance one (the interesting one).** Section 4
selects pairs by detection. Replace that rule: keep every candidate pair where either gene is
in a list of immune genes you write yourself &mdash; start from
`["PTPRC", "CD3D", "CD4", "CD8A", "CD68", "CD163", "HLA-DRA", "CXCL9", "CXCL13", "CCL19",
"CD274", "PDCD1", "CTLA4", "CD28", "ITGAL", "ITGB2"]` &mdash; and re-run. You will test far
fewer pairs, so it will be faster, and the top of the list will be a completely different set
of biology. Which version would you put in a paper, and why?

**3. Use the putative database.** `connectomeDB2020_put` has 4,071 pairs against `lit`'s 2,293.
Swap it in at Section 2 and re-run. How many extra pairs clear the 5% threshold, and do any of
them displace the literature-supported pairs at the top? Anything that does deserves five
minutes with a search engine.

**4. Cross-check against Tutorial 2's niches.** Re-run Tutorial 2's neighbourhood-composition
KMeans on these cells (the code is fifteen lines) to get a `niche` label, then pass
`use_label="niche"` to `run_cci` instead of `cell_type`. You now have signalling between
*niches* rather than between types. Does the immune-infiltrate niche receive from the tumour
niche, or only from the stroma?

**5. Break it on purpose.** Shuffle `adata.obs["cell_type"]` with `np.random.permutation`,
keeping coordinates and expression fixed, and re-run **Section 7 only**. `run_cci` should
collapse to nothing, because the permutation null is built by shuffling exactly those labels.
Now instead shuffle the *coordinates* and re-run **Section 5**. If `cci.run` still returns
hundreds of significant pairs, the spatial constraint is not doing what you think it is. Both
controls take two minutes and neither is optional in real work.

**6. Follow one pair all the way down.** Pick a pair from Section 7.2, find the cells where it
is significant (`adata.obsm["lr_sig_scores"]`), and cross-tabulate those cells' types against
the rest of the Crop. Then plot them on the H&E with `score_map`. Can you write one sentence
about this tumour that you would be willing to defend?

---

## Further reading

**The method used here.** [Pham *et al.*, "Robust mapping of spatiotemporal trajectories and
cell&ndash;cell interactions in healthy and diseased tissues", *Nature Communications* 14, 7739
(2023)](https://www.nature.com/articles/s41467-023-43120-6) &mdash; the stLearn paper. The
cell&ndash;cell interaction section describes the permutation scheme the two analysis cells
above run, including why the background is matched on expression rather than drawn uniformly.

**The pair list.** [Hou *et al.*, "Predicting cell-to-cell communication networks using
NATMI", *Nature Communications* 11, 5011 (2020)](https://www.nature.com/articles/s41467-020-18873-z)
&mdash; connectomeDB2020, its curation criteria, and the literature-versus-putative
distinction that Section 2 leans on.

**Other tools for the same question.**

* [**CellPhoneDB**](https://www.nature.com/articles/s41596-020-0292-x) (Nature Protocols, 2020)
  &mdash; the most widely used ligand&ndash;receptor method, and the one your reviewers will
  know. Non-spatial by default: it permutes cluster labels rather than using positions.
  `squidpy.gr.ligrec` runs the same statistic and is already installed by Tutorial 2.
* [**CellChat**](https://www.nature.com/articles/s41467-021-21246-8) (Nature Communications,
  2021) &mdash; R, and stronger than anything here at *organising* results: pairs are grouped
  into signalling pathways, so you read "TGFb signalling" instead of eleven separate
  TGFB1&ndash;receptor rows.
* [**NicheNet**](https://www.nature.com/articles/s41592-019-0667-5) (Nature Methods, 2020)
  &mdash; asks the harder question. Rather than stopping at co-expression, it links a ligand to
  the *downstream transcriptional response* it should induce in the receiver, so a prediction
  becomes falsifiable within the same dataset.
* [**COMMOT**](https://www.nature.com/articles/s41592-022-01728-4) (Nature Methods, 2023)
  &mdash; treats signalling as optimal transport over the tissue, which handles competition
  between receivers for a limited ligand supply, something none of the counting methods do.

### Where this Tutorial sits

Tutorial 1 read three platforms and put them on screen. Tutorial 2 asked which cell types sit
together and turned the answer into niches you could measure. This Tutorial took the same
cells, swapped the panel, and asked what might be passing between them &mdash; and the
tumour&ndash;stroma interfaces it found are the same interfaces Tutorial 2's niches drew, which
is the strongest kind of agreement two methods can offer. Tutorial 3 leaves transcripts behind
entirely and asks what could have been inferred from the H&E image alone.

The through-line: **every one of these methods has a boundary, and the boundary usually lands
on the immune compartment.** Tutorial 2 could not resolve a B cell cluster. This Tutorial ranks
every immune pair below the extracellular matrix. Tutorial 3 predicts stromal genes from
morphology and immune genes badly. Three different methods, three different failure modes, one
consistent blind spot &mdash; and knowing where it lands is the difference between using these
tools and being used by them.